In [0]:
%sql
USE CATALOG taxi_data_bronze

Create Table schema to ingest the raw data as it is in the original source.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS taxi_data_bronze.data_ingestion.trips
(
  tpep_pickup_datetime TIMESTAMP,
  tpep_dropoff_datetime TIMESTAMP,
  trip_distance DOUBLE, 
  fare_amount DOUBLE,
  pickup_zip INT,
  dropoff_zip INT
)


Read the sourse data into a dataframe to be written on to the schema table

In [0]:
nyc_sample_data_path = "samples.nyctaxi.trips"

ingest_df = spark.table(nyc_sample_data_path)


In [0]:
# Since the project only reads the data once, we can write the data using "overwrite" mode

ingest_df.write.mode("overwrite").saveAsTable("taxi_data_bronze.data_ingestion.trips")

Following codes are to test various attributes and accuracies of the data

In [0]:
trips_data_test = spark.table("taxi_data_bronze.data_ingestion.trips")

display(trips_data_test.count())

In [0]:
%sql
-- Explore the max and min values of the variables
select 
  max(t.tpep_dropoff_datetime) AS max_dropoff,
  min(t.tpep_dropoff_datetime) AS min_dropoff,
  min(t.tpep_pickup_datetime) AS min_pickup,
  max(t.tpep_pickup_datetime) AS max_pickup,
  max(t.trip_distance) AS max_distance,
  min(t.trip_distance) AS min_distance,
  max(t.fare_amount) AS max_fare,
  min(t.fare_amount) AS min_fare,
  max(t.pickup_zip) AS max_pickup_zip,
  min(t.pickup_zip) AS min_pickup_zip,
  max(t.dropoff_zip) AS max_dropoff_zip,
  min(t.dropoff_zip) AS min_dropoff_zip,
  count(*) AS total_trips
from taxi_data_bronze.data_ingestion.trips AS t;



In [0]:
%sql
-- Check for bad data - no distance, no fare, negative distance, negative fare,
    SELECT count(t.fare_amount) AS bad_fare_amount,
      (SELECT Count(*) 
      FROM taxi_data_bronze.data_ingestion.trips AS t
      WHERE 
          trip_distance <= 0 OR trip_distance IS NULL) AS bad_trip_distance
    FROM taxi_data_bronze.data_ingestion.trips AS t
    WHERE fare_amount <= 0 OR fare_amount IS NULL;